# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M3 — Document Ingestion, Chunking, Indexing and Memory

### Goal
Prepare the CareFlow policy corpus for retrieval.

**Primary asset:** `data/careflow/corpus/markdown/*.md` and `data/careflow/corpus/pdf/*.pdf`.

The corpus should contain roughly 14 operational documents. Do not allow generated documents to become a source of diagnostic or treatment advice.

In [4]:
from pathlib import Path
import re, math
from collections import Counter, defaultdict

CORPUS_RELATIVE_PATH = Path("data/careflow/corpus/markdown")
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / CORPUS_RELATIVE_PATH).is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot find {CORPUS_RELATIVE_PATH} from notebook working directory {Path.cwd()}"
    )

CORPUS = PROJECT_ROOT / CORPUS_RELATIVE_PATH
docs = [
    {"source": p.name, "text": p.read_text(encoding="utf-8")}
    for p in sorted(CORPUS.glob("*.md"))
]
if not docs:
    raise FileNotFoundError(f"No Markdown documents found in {CORPUS.resolve()}")

print("Corpus directory:", CORPUS.resolve())
print("Documents loaded:", len(docs))

Corpus directory: /Users/sgunaseelan/Projects/API/AI-IIT/applied-ai-capstone-starter/data/careflow/corpus/markdown
Documents loaded: 14


## Chunking strategy

Use section-aware chunks first. Preserve:

- source document
- heading/section
- chunk id
- text

This metadata becomes part of retrieval provenance.

In [5]:
def chunk_text(text, max_words=180, overlap=35):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + max_words)
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = max(0, end - overlap)
    return chunks

chunks = []
for d in docs:
    for i, c in enumerate(chunk_text(d["text"])):
        chunks.append({"chunk_id": f'{d["source"]}::chunk-{i:03d}', "source": d["source"], "text": c})

print("Chunks:", len(chunks))

Chunks: 89


In [6]:
# Lightweight lexical index. This is an intentionally dependency-free baseline.
STOP = {"the","and","for","with","that","this","from","are","was","were","what","how","which","into","under"}

def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9-]+", text.lower()) if w not in STOP]

index = [set(tokenize(c["text"])) for c in chunks]

def lexical_search(query, k=5):
    q = set(tokenize(query))
    scored = []
    for c, terms in zip(chunks, index):
        score = len(q & terms)
        if score:
            scored.append((score, c))
    return sorted(scored, key=lambda x: x[0], reverse=True)[:k]

for score, item in lexical_search("specialist co-pay MERIDIAN-GOLD", 3):
    print(score, item["source"], item["chunk_id"])

3 copay-schedule.md copay-schedule.md::chunk-000
2 copay-schedule.md copay-schedule.md::chunk-001
2 copay-schedule.md copay-schedule.md::chunk-002


## Memory/index design

For M3, the index is the long-lived memory of policy knowledge:

```text
document -> chunks -> metadata -> index
```

Later milestones can replace the lexical baseline with embeddings/vector search without changing the source contract.